In [ ]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
import faiss
import time
import gc
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn
import warnings

warnings.filterwarnings('ignore')

# Detect hardware
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CPU_THREADS = 30  # Leaving 2 vCPUs for OS stability

print(f"Device: {DEVICE} ({torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'})")
print(f"Allocated Threads: {CPU_THREADS} vCPUs | System RAM Ceiling: 128 GiB")

# ==============================================================================
# 1. DATA LOADING & INT32 INDEX MAPPING
# ==============================================================================
print("\n1. Loading Data & Building Integer Index Map...")
cols = ['entity_id', 'clean_name', 'Trans_name', 'norm_address', 'country']

s1 = pd.read_csv("../ML2026/cleaned_dataset/source1.tsv", sep='\t', usecols=cols).fillna('')
target_df = pd.concat([
    pd.read_csv("../ML2026/cleaned_dataset/source2.tsv", sep='\t', usecols=cols).fillna(''),
    pd.read_csv("../ML2026/cleaned_dataset/source3.tsv", sep='\t', usecols=cols).fillna('')
], ignore_index=True)

s1['country'] = s1['country'].replace('', 'UNKNOWN_COUNTRY')
target_df['country'] = target_df['country'].replace('', 'UNKNOWN_COUNTRY')

# Retain original ID arrays for zero-copy mapping at export
s1_id_lookup = s1['entity_id'].values
tgt_id_lookup = target_df['entity_id'].values

# Assign lightweight int32 indices to avoid string-join RAM spikes
s1['s1_idx'] = np.arange(len(s1), dtype=np.int32)
target_df['tgt_idx'] = np.arange(len(target_df), dtype=np.int32)

# Helper feature generators
def sort_string(text):
    return " ".join(sorted(str(text).split()))

def extract_digits(text):
    digits = re.sub(r'\D+', '', str(text))
    return digits if len(digits) >= 6 else ''

def make_acronym(text):
    words = str(text).split()
    return "".join([w[0] for w in words]) if len(words) > 1 else ''

s1['sorted_name'] = s1['clean_name'].apply(sort_string)
target_df['sorted_name'] = target_df['clean_name'].apply(sort_string)

s1['addr_digits'] = s1['norm_address'].apply(extract_digits)
target_df['addr_digits'] = target_df['norm_address'].apply(extract_digits)

s1['acronym'] = s1['clean_name'].apply(make_acronym)
target_df['acronym'] = target_df['clean_name'].apply(make_acronym)

candidate_blocks = []

# ==============================================================================
# 2. SINGLE-GPU T4 SEMANTIC BLOCKING
# ==============================================================================
print("\n2. Executing Semantic Blocking on NVIDIA T4...")
start_sem = time.time()

unique_s1_names = s1['clean_name'].unique()
unique_tgt_names = target_df['clean_name'].unique()

print(f" -> Encoding {len(unique_tgt_names):,} Target & {len(unique_s1_names):,} S1 Unique Names...")
model = SentenceTransformer('BAAI/bge-small-en-v1.5', device=DEVICE)

# 1024 batch size saturates 16GB VRAM cleanly
tgt_embeddings = model.encode(
    unique_tgt_names, 
    batch_size=1024, 
    show_progress_bar=True, 
    normalize_embeddings=True, 
    convert_to_numpy=True
)
s1_embeddings = model.encode(
    unique_s1_names, 
    batch_size=1024, 
    show_progress_bar=True, 
    normalize_embeddings=True, 
    convert_to_numpy=True
)

del model
torch.cuda.empty_cache()
gc.collect()

In [30]:
import joblib

joblib.dump(tgt_embeddings, 'tgt_embeddings.pkl')
joblib.dump(s1_embeddings, 's1_embeddings.pkl')

['s1_embeddings.pkl']

In [3]:
!pip uninstall -y faiss-cpu faiss-gpu
!conda install -y -c pytorch -c nvidia faiss-gpu

Found existing installation: faiss-gpu 1.15.1
Uninstalling faiss-gpu-1.15.1:
  Successfully uninstalled faiss-gpu-1.15.1
Retrieving notices: done
Channels:
 - pytorch
 - nvidia
 - conda-forge
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /opt/conda

  added / updated specs:
    - faiss-gpu


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    conda-26.7.2               |  py312h20c3967_2         1.3 MB  conda-forge
    faiss-1.14.3               |py312cuda129h2701306_501_cuda         1.8 MB  conda-forge
    faiss-gpu-1.14.3           |     h92135e1_501          18 KB  conda-forge
    libfaiss-1.14.3            |cuda129h26894d9_501_cuda        91.4 MB  conda-forge
    ------------------------------------------------------------
                                           Total:        94.5 MB

The following NEW packages will be INSTALLED:

  faiss   

In [8]:
torch.cuda.empty_cache()


In [ ]:
import faiss
import joblib
from tqdm import tqdm

tgt_embeddings = joblib.load('tgt_embeddings.pkl')
s1_embeddings = joblib.load('s1_embeddings.pkl') 

print(" -> Initializing FAISS GPU with FP16 Memory Compression...")
dim = tgt_embeddings.shape[1]
res = faiss.StandardGpuResources()

# Configure FAISS to store the index in 16-bit floats to prevent OOM
config = faiss.GpuIndexFlatConfig()
config.device = 0
config.useFloat16 = True  # Cuts VRAM requirement from 11.4 GiB to 5.7 GiB

# Create the GPU index directly
gpu_index = faiss.GpuIndexFlatIP(res, dim, config)

print(" -> Adding Target Embeddings to FAISS Index...")
gpu_index.add(tgt_embeddings)

print(" -> Executing Top-5 Nearest Neighbor Search in batches...")
distances_list, indices_list = [], []

# Batching the queries prevents FAISS from allocating massive C++ workspace memory
batch_size = 50000 
for i in tqdm(range(0, len(s1_embeddings), batch_size), desc='FAISS Searching'):
    batch = s1_embeddings[i:i+batch_size]
    batch_dist, batch_ind = gpu_index.search(batch, 5)
    
    distances_list.append(batch_dist)
    indices_list.append(batch_ind)

In [6]:
distances_list


NameError: name 'distances_list' is not defined

In [4]:
import numpy as np
import torch
distances = np.vstack(distances_list)
indices = np.vstack(indices_list)

# Clear GPU memory aggressively
del gpu_index, res, tgt_embeddings, s1_embeddings, distances_list, indices_list
torch.cuda.empty_cache()
gc.collect()

print(" -> Mapping Semantic Pairs back to Integer Indices...")
# Fast dictionary lookup for index arrays
s1_name_to_idx = s1.groupby('clean_name')['s1_idx'].apply(list).to_dict()
tgt_name_to_idx = target_df.groupby('clean_name')['tgt_idx'].apply(list).to_dict()

unique_s1_names = s1['clean_name'].unique()
unique_tgt_names = target_df['clean_name'].unique()

sem_s1_idx, sem_tgt_idx, sem_scores = [], [], []
for i, s1_name in tqdm(enumerate(unique_s1_names), desc='Mapping IDs'):
    for j in range(5):
        if distances[i][j] >= 0.90:
            tgt_name = unique_tgt_names[indices[i][j]]
            for s1_i in s1_name_to_idx.get(s1_name, []):
                for tgt_i in tgt_name_to_idx.get(tgt_name, []):
                    sem_s1_idx.append(s1_i)
                    sem_tgt_idx.append(tgt_i)
                    sem_scores.append(distances[i][j])

df_sem = pd.DataFrame({
    's1_idx': np.array(sem_s1_idx, dtype=np.int32),
    'tgt_idx': np.array(sem_tgt_idx, dtype=np.int32),
    'confidence_score': np.array(sem_scores, dtype=np.float32)
})
candidate_blocks.append(df_sem)

print(f"Semantic Blocking completed in {time.time() - start_sem:.2f}s.")
del  df_sem, s1_name_to_idx, tgt_name_to_idx
del sem_s1_idx, sem_tgt_idx, sem_scores, distances, indices
gc.collect()

NameError: name 'distances_list' is not defined

In [ ]:
# ==============================================================================
# 3. EXACT HASH JOINS (INT32 COMPRESSED)
# ==============================================================================
print(f"\n3. Executing O(1) Hash Joins on {CPU_THREADS} Cores...")

def hash_join(df1, df2, keys, conf):
    m = df1[df1[keys[-1]] != ''][['s1_idx'] + keys].merge(
        df2[df2[keys[-1]] != ''][['tgt_idx'] + keys], on=keys
    )[['s1_idx', 'tgt_idx']]
    m['confidence_score'] = np.float32(conf)
    return m

candidate_blocks.append(hash_join(s1, target_df, ['country', 'clean_name'], 1.0))
candidate_blocks.append(hash_join(s1, target_df, ['country', 'Trans_name'], 1.0))
candidate_blocks.append(hash_join(s1, target_df, ['country', 'norm_address'], 1.0))
candidate_blocks.append(hash_join(s1, target_df, ['country', 'sorted_name'], 0.9))
candidate_blocks.append(hash_join(s1, target_df, ['country', 'addr_digits'], 0.8))

# Acronym Cross-Matches
m_acr1 = s1[s1['acronym'] != ''][['s1_idx', 'country', 'acronym']].merge(
    target_df[target_df['clean_name'] != ''][['tgt_idx', 'country', 'clean_name']],
    left_on=['country', 'acronym'], right_on=['country', 'clean_name']
)[['s1_idx', 'tgt_idx']]
m_acr1['confidence_score'] = np.float32(0.7)
candidate_blocks.append(m_acr1)

m_acr2 = s1[s1['clean_name'] != ''][['s1_idx', 'country', 'clean_name']].merge(
    target_df[target_df['acronym'] != ''][['tgt_idx', 'country', 'acronym']],
    left_on=['country', 'clean_name'], right_on=['country', 'acronym']
)[['s1_idx', 'tgt_idx']]
m_acr2['confidence_score'] = np.float32(0.7)
candidate_blocks.append(m_acr2)

del m_acr1, m_acr2
gc.collect()

In [ ]:
# ==============================================================================
# 4. TF-IDF SPARSE DOT PRODUCT (30 THREADS, MEMORY CAPPED)
# ==============================================================================
print(f"\n4. Executing Sparse Dot Products...")

def tfidf_block(col, thresh, top_n, analyzer, n_gram, max_feat):
    vec = TfidfVectorizer(
        analyzer=analyzer, 
        ngram_range=n_gram, 
        max_features=max_feat, 
        dtype=np.float32
    )
    mat_tgt = vec.fit_transform(target_df[col].values).transpose().tocsr()
    mat_s1 = vec.transform(s1[col].values)
    
    matches = sp_matmul_topn(mat_s1, mat_tgt, top_n=top_n, threshold=thresh, n_threads=CPU_THREADS)
    non_zeros = matches.nonzero()
    
    return pd.DataFrame({
        's1_idx': s1['s1_idx'].values[non_zeros[0]],
        'tgt_idx': target_df['tgt_idx'].values[non_zeros[1]],
        'confidence_score': matches.data.astype(np.float32)
    })

candidate_blocks.append(tfidf_block('clean_name', 0.50, 20, 'word', (1,2), None))
candidate_blocks.append(tfidf_block('norm_address', 0.60, 20, 'word', (1,2), None))
candidate_blocks.append(tfidf_block('clean_name', 0.45, 30, 'char_wb', (3,4), 150000))

In [1]:
# ==============================================================================
# 5. IN-PLACE DEDUPLICATION & EXPORT
# ==============================================================================
print("\n5. Consolidating & Deduplicating Integer Pairs...")
final_candidates = pd.concat(candidate_blocks, ignore_index=True)

del candidate_blocks, s1, target_df
gc.collect()

# Sorting and dropping duplicates on int32 + float32 is light on RAM
final_candidates.sort_values('confidence_score', ascending=False, inplace=True)
final_candidates.drop_duplicates(subset=['s1_idx', 'tgt_idx'], keep='first', inplace=True)

print(f"Total Unique Pairs: {len(final_candidates):,}")
print("Mapping Integer IDs back to original Entity IDs for export...")

# Map back directly via low-overhead numpy indexing
final_candidates['s1_entity_id'] = s1_id_lookup[final_candidates['s1_idx'].values]
final_candidates['target_entity_id'] = tgt_id_lookup[final_candidates['tgt_idx'].values]

final_candidates.drop(columns=['s1_idx', 'tgt_idx'], inplace=True)

# Write to disk
final_candidates.to_csv("stage1_hybrid_candidates.csv", index=False)
print("Pipeline Complete! Output saved to 'stage1_hybrid_candidates.csv'")

 -> Bypassing FAISS: Running Exact Nearest Neighbors natively via PyTorch...


RuntimeError: expected mat1 and mat2 to have the same dtype, but got: float != c10::Half